# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a walkthrough for loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing schema entities by their `@id` throughout. All entities (record sets, fields, columns) are referenced by their Croissant `@id`, ensuring consistent and reproducible access to data.

### Dataset Source
Croissant schema: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets. For each, present its `@id` and the available fields (and their `@id`s), as defined in the Croissant schema.

**Note:** All references are by `@id`.

In [ ]:
# List all record sets in the dataset by their @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets declared in metadata; attempting to infer from schema.")

# If no record set defined in the metadata, try to fetch all top-level record sets
all_record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None) for rs in dataset.record_sets]
# The above usually gives an empty list - let's try to directly explore records in this dataset by schema insight
print('Available record sets:')
for rs in dataset.list_record_sets():
    print(f"  - {rs['@id']} (name: {rs['name']})")

# For each record set, show available fields and columns by @id
for rs in dataset.list_record_sets():
    print(f"\nRecord set: {rs['@id']} (name: {rs['name']})")
    fields = rs.get('field', [])
    if fields:
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            fid = field.get('@id', None)
            fname = field.get('name', '')
            print(f"    - {fid} (name: {fname})")
            # Show columns if present
            columns = field.get('column', [])
            if columns:
                if isinstance(columns, dict):
                    columns = [columns]
                for column in columns:
                    colid = column.get('@id', None)
                    colname = column.get('name', '')
                    print(f"       column: {colid} (name: {colname})")
    else:
        print("  No explicit fields defined.")

## 3. Data Extraction

Load all records from a specific record set into a DataFrame for analysis. Use `@id` from the record set overview above, as well as field `@id`s.

In [ ]:
# Find all record set @id's programmatically
record_set_ids = [rs['@id'] for rs in dataset.list_record_sets()]

if not record_set_ids:
    raise RuntimeError("No record sets found in this Croissant schema.")

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        print(f"  No records found for record set {record_set_id}.")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  DataFrame columns: {df.columns.tolist()}")
    display(df.head(3))

# For demonstration, pick the first record set (if any records exist)
main_record_set_id = next(iter(dataframes.keys())) if dataframes else None
if main_record_set_id:
    print(f"Main record set selected: {main_record_set_id}")
    print(f"Sample fields: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No records data available in any record set.")

## 4. Exploratory Data Analysis (EDA)

Let us conduct data processing and simple analysis. We'll select numeric fields and categorical fields by their `@id`s.
Typical EDA operations include filtering, normalization, and grouping--all referencing fields by `@id`.

In [ ]:
import numpy as np

# Use the main loaded record set and identify numeric fields
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Available fields in DataFrame: {df.columns.tolist()}")

    # Try to programmatically select a numeric field by name or @id
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()

    # If no numeric columns, try to infer from columns names (e.g., 'Age', 'interval', etc.)
    if not numeric_candidates:
        numeric_candidates = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower()]
    if not numeric_candidates:
        print("No obvious numeric fields found.")
    else:
        numeric_field_id = numeric_candidates[0]
        print(f"Analyzing numeric field (referenced by @id or column name): {numeric_field_id}")

        # Convert to numeric, forcing errors to NaN
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].quantile(0.25)  # Use lower quartile as threshold for demonstration

        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Attempt grouping by the first non-numeric, suitable categorical field
        group_candidates = df.select_dtypes(include='object').columns.tolist()
        group_field_id = None
        for col in group_candidates:
            if df[col].nunique() < len(df) / 2 and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
else:
    print("No main record set DataFrame was loaded.")

## 5. Visualization

Visualize the distribution of the selected numeric field and its relationship to the grouping field (if available).
All column/field references are made by their `@id`s or dataset column names.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and 'numeric_field_id' in locals():
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color='skyblue')
    plt.xlabel(numeric_field_id)
    plt.title(f'Distribution of {numeric_field_id} (@id or column)')
    plt.show()

    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} distribution by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough data for visualization. Load records first.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore the FAIR^2 colorectal cancer survivors dataset using `mlcroissant`, referencing all fields, columns, and record sets by their schema `@id` as per best practices. Further domain-specific analyses (e.g., survival analysis, molecular subtyping) can be performed building on this reproducible Croissant pipeline.